# Restoration Benchmark Evaluation

This notebook evaluates model predictions for one benchmark task under three prompting strategies: **Zero-Shot**, **CoT**, and **CoT+ICL**.

**Metrics**

| Metric | Text form | Description |
|---|---|---|
| Accuracy (%) | compact | Exact match with any reference answer |
| Edit Distance | compact | Minimum Levenshtein distance to any reference answer |
| Sentence-BERT | spaced | Maximum cosine similarity to any reference answer |
| chrF++ | spaced | Mean sentence-level chrF++ (`char_order=6`, `word_order=2`, `beta=2`) |

**Tasks**

| English | Korean |
|---|---|
| `eng_consonant_vowel_combination` | `kor_consonant_vowel_combination` |
| `eng_visual_transform` | `kor_visual_transform` |
| `eng_abbreviations` | `kor_abbreviations` |
| `eng_phonetic_substitution` | `kor_letter_rotation` |

**Requirements**

Python 3.10 or later.

```
pip install pandas numpy python-Levenshtein nltk sentence-transformers sacrebleu==2.5.1
```

The Sentence-BERT models and, for `eng_consonant_vowel_combination`, the NLTK word list are downloaded on first use.

## 1. Configuration

Set the task name and the paths to the gold file and the three prediction files, then run all cells.

**Gold file**: a JSON list with one object per sample, containing the reference answer (`original`) and the transformed input (`transformed`). Multiple acceptable answers are separated by `|`.

```json
[{"id": 0, "original": "elite", "transformed": "1337"}, ...]
```

**Prediction files**: a JSON list in the same order as the gold file, with the model answer in `Answer`. If a `word` field holds the model input, the notebook checks that the rows follow the gold order.

```json
[{"id": 0, "word": "1337", "Answer": "elite"}, ...]
```

In [ ]:
TASK = ""            # e.g., "eng_consonant_vowel_combination"
DATA_ORIGINAL = ""   # gold file, e.g., "data/eng_consonant_vowel_combination.json"
DATA_ZRS = ""        # Zero-Shot predictions
DATA_COT = ""        # CoT predictions
DATA_COT_ICL = ""    # CoT+ICL predictions

OUTPUT_CSV = ""      # optional: path to save the summary table (leave empty to skip)

# Sentence-BERT model per language
SBERT_MODELS = {
    "eng": "sentence-transformers/all-MiniLM-L6-v2",
    "kor": "jhgan/ko-sroberta-multitask",
}

## 2. Setup

In [ ]:
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from Levenshtein import distance as levenshtein
from sacrebleu.metrics import CHRF
from sentence_transformers import SentenceTransformer

TASKS = (
    "eng_consonant_vowel_combination", "eng_visual_transform",
    "eng_abbreviations", "eng_phonetic_substitution",
    "kor_consonant_vowel_combination", "kor_visual_transform",
    "kor_abbreviations", "kor_letter_rotation",
)
if TASK not in TASKS:
    raise ValueError(f"TASK must be one of {TASKS}, got {TASK!r}.")

LANGUAGE = "kor" if TASK.startswith("kor_") else "eng"
SBERT_MODEL = SBERT_MODELS[LANGUAGE]
PROMPTS = {"ZRS": DATA_ZRS, "COT": DATA_COT, "COT_ICL": DATA_COT_ICL}

print(f"Task : {TASK}")
print(f"SBERT: {SBERT_MODEL}")

## 3. Load data

Rows are paired by position. Field names are matched in the order listed in `get_field`, so the benchmark files in `Dataset/` (`original`/`transformed`, `words`/`random`, `word`/`leetspeak`) are all accepted.

In [ ]:
def load_json(path):
    with open(path, encoding="utf-8-sig") as file:
        return json.load(file)


def get_field(row, keys):
    for key in keys:
        if key in row:
            return row[key]
    raise KeyError(f"Expected one of {keys}; available fields: {list(row)}")


def order_key(text):
    """Key for the row-order check: NFC, lowercase, whitespace removed."""
    text = unicodedata.normalize("NFC", str(text)).lower()
    return re.sub(r"\s+", "", text)


gold = load_json(DATA_ORIGINAL)
df = pd.DataFrame({
    "original": [get_field(row, ("original", "words", "word")) for row in gold],
    "transformed": [get_field(row, ("transformed", "random", "leetspeak")) for row in gold],
})

for prompt, path in PROMPTS.items():
    rows = load_json(path)
    if len(rows) != len(gold):
        raise ValueError(f"{prompt}: {len(rows)} predictions for {len(gold)} gold rows.")

    inputs = [row.get("word") for row in rows]
    if all(value is not None for value in inputs):
        mismatched = sum(
            order_key(value) != order_key(transformed)
            for value, transformed in zip(inputs, df["transformed"])
        )
        if mismatched:
            raise ValueError(f"{prompt}: {mismatched} rows do not follow the gold order.")

    df[f"{prompt}_answer"] = [
        get_field(row, ("Answer", "answer", "output", "prediction")) for row in rows
    ]

print(f"Loaded {len(df)} samples.")

## 4. Text normalization and references

All text is NFC-normalized and lowercased, and every character other than Latin letters, digits, and Hangul (including jamo) is replaced by a space.

- **compact** (Accuracy, Edit Distance): all spaces are removed, so spacing differences are not counted as errors.
- **spaced** (Sentence-BERT, chrF++): single spaces between words are kept to retain word boundaries.

A reference containing `|` is split into alternative answers; a prediction is scored against the best-matching alternative.

In [ ]:
def normalize(text):
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[^a-z0-9ㄱ-ㅎㅏ-ㅣ가-힣 ]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_text(text):
    return normalize(text).replace(" ", "")


def spaced_text(text):
    return normalize(text)


def split_references(value):
    parts = value if isinstance(value, list) else [value]
    references = [
        reference.strip()
        for part in parts
        if part is not None
        for reference in str(part).split("|")
        if reference.strip()
    ]
    if not references:
        raise ValueError("A gold row has no reference answer.")
    return references


references = df["original"].map(split_references).tolist()
compact_references = [[compact_text(ref) for ref in refs] for refs in references]
spaced_references = [[spaced_text(ref) for ref in refs] for refs in references]

predictions = {
    prompt: [spaced_text("" if text is None else str(text)) for text in df[f"{prompt}_answer"]]
    for prompt in PROMPTS
}

## 5. Accuracy rule for English consonant–vowel combination

In `eng_consonant_vowel_combination`, the input is a scrambled arrangement of the letters of the target word. A prediction is therefore also counted as correct for **Accuracy** if it

1. uses exactly the same letters as the reference (an anagram of the reference), and
2. is a valid English word in the NLTK `words` corpus.

This rule applies only to Accuracy in this task. Edit Distance, Sentence-BERT, and chrF++ always compare the prediction with the original reference answer.

In [ ]:
english_words = set()
if TASK == "eng_consonant_vowel_combination":
    import nltk
    from nltk.corpus import words

    try:
        english_words = set(words.words())
    except LookupError:
        nltk.download("words", raise_on_error=True)
        english_words = set(words.words())


def is_correct(prediction, reference):
    """Exact match on compact text, plus the dictionary-valid anagram rule for English C-V."""
    if not prediction:
        return False
    if prediction == reference:
        return True
    return (
        prediction in english_words
        and sorted(prediction) == sorted(reference)
    )

## 6. Evaluate

In [ ]:
chrf = CHRF(char_order=6, word_order=2, beta=2)
sbert = SentenceTransformer(SBERT_MODEL)

# Encode every distinct reference and prediction once.
texts = dict.fromkeys(ref for refs in spaced_references for ref in refs if ref)
for answers in predictions.values():
    texts.update(dict.fromkeys(answer for answer in answers if answer))
vectors = sbert.encode(list(texts), batch_size=64, normalize_embeddings=True,
                       show_progress_bar=True)
embeddings = dict(zip(texts, vectors))


def sbert_score(prediction, refs):
    refs = [ref for ref in refs if ref]
    if not prediction or not refs:
        return 0.0
    return max(float(embeddings[prediction] @ embeddings[ref]) for ref in refs)


summary = {}
for prompt, answers in predictions.items():
    compact_answers = [compact_text(answer) for answer in answers]
    scores = {
        "Accuracy(%)": [
            100 * any(is_correct(answer, ref) for ref in refs)
            for answer, refs in zip(compact_answers, compact_references)
        ],
        "Edit Distance": [
            min(levenshtein(answer, ref) for ref in refs)
            for answer, refs in zip(compact_answers, compact_references)
        ],
        "SBERT Cosine Similarity": [
            sbert_score(answer, refs)
            for answer, refs in zip(answers, spaced_references)
        ],
        "chrF++ Sentence Avg": [
            chrf.sentence_score(answer, refs).score
            for answer, refs in zip(answers, spaced_references)
        ],
    }
    for metric, values in scores.items():
        df[f"{prompt}_{metric}"] = values

    summary[prompt] = {metric: float(np.mean(values)) for metric, values in scores.items()}

df_performance = pd.DataFrame.from_dict(summary, orient="index")
df_performance.index.name = "Prompt"
df_performance["SBERT model"] = SBERT_MODEL

if OUTPUT_CSV:
    df_performance.round(4).to_csv(OUTPUT_CSV, encoding="utf-8-sig")
    print(f"Saved: {OUTPUT_CSV}")

df_performance.round(4)

## 7. Per-sample results (optional)

`df` holds each sample's predictions and scores, which is useful for error analysis.

In [ ]:
df.head(10)